# Vectorbt Indicator Optimizer Test

Test notebook for the new vectorbt-based indicator optimization module.

In [1]:
# Setup - suppress autoreload numpy warnings
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

# Set logging
import logging
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')

## 1. Test Single Indicator Optimization

In [5]:
from crypto_analysis.vectorbt_optimizer import optimize_indicator

# Test RSI optimization for BTC
result = optimize_indicator("RSI", "BTC")

print(f"RSI optimization for BTC:")
print(f"  Best params: {result.best_params}")
print(f"  Score (total return): {result.score:.4f}")
print(f"  Sampler type: {result.sampler_type}")
print(f"  Trials: {result.n_trials}")
if result.fitness_details:
    print(f"  Fitness details: {result.fitness_details}")

crypto_analysis.vectorbt_optimizer.optimizer - INFO - Optimizing RSI: 12789 combinations
crypto_analysis.vectorbt_optimizer.optimizer - INFO - Using tpe sampler with 1000 trials
crypto_analysis.vectorbt_optimizer - INFO - Optimized RSI for BTC: score=0.6401, params={'timeperiod': 28, 'entry_constant': 29.19571220333563, 'exit_constant': 63.86964289663444}


RSI optimization for BTC:
  Best params: {'timeperiod': 28, 'entry_constant': 29.19571220333563, 'exit_constant': 63.86964289663444}
  Score (total return): 0.6401
  Sampler type: tpe
  Trials: 1000
  Fitness details: FitnessResult(return=0.6401, trades=14, win_rate=78.57%, sharpe=2.06)


## 2. Test Data Loader

In [6]:
from crypto_analysis.vectorbt_optimizer import DataLoader, list_available_symbols, load_whitelist
from pathlib import Path

# List available symbols
symbols = list_available_symbols("../data/binance")
print(f"Available symbols ({len(symbols)}): {symbols}")

# Load whitelist
whitelist = load_whitelist("../config.json")
print(f"\nWhitelist symbols ({len(whitelist)}): {whitelist}")

Available symbols (30): ['ADA', 'ALGO', 'ARB', 'ATOM', 'AVAX', 'BNB', 'BTC', 'CHZ', 'DOGE', 'DOT', 'ETH', 'FIL', 'HBAR', 'ICP', 'IOTA', 'LDO', 'LINK', 'LRC', 'LTC', 'NEAR', 'ONE', 'OP', 'PEPE', 'SHIB', 'SOL', 'TRX', 'VET', 'XLM', 'XRP', 'ZIL']

Whitelist symbols (31): ['BTC', 'ETH', 'BNB', 'SOL', 'XRP', 'ADA', 'DOT', 'LINK', 'AVAX', 'MATIC', 'ATOM', 'NEAR', 'LTC', 'ARB', 'OP', 'LRC', 'TRX', 'XLM', 'DOGE', 'VET', 'ALGO', 'HBAR', 'FIL', 'EOS', 'CHZ', 'IOTA', 'CRO', 'ZIL', 'ONE', 'LDO', 'ICP']


In [7]:
# Load sample data
from crypto_analysis.vectorbt_optimizer import load_feather

df = load_feather("BTC", "../data/binance")
print(f"BTC data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

BTC data shape: (9208, 6)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume']
Date range: 2025-01-01 00:00:00+00:00 to 2026-01-19 15:00:00+00:00


,date,open,high,low,close,volume
0,2025-01-01 00:00:00+00:00,93576.00,94509.42,93489.03,94401.14,755.99010
1,2025-01-01 01:00:00+00:00,94401.13,94408.72,93578.77,93607.74,586.53456
2,2025-01-01 02:00:00+00:00,93607.74,94105.12,93594.56,94098.91,276.78045
3,2025-01-01 03:00:00+00:00,94098.90,94098.91,93728.22,93838.04,220.99302
4,2025-01-01 04:00:00+00:00,93838.04,93838.04,93500.00,93553.91,279.46909


## 3. Test Vectorbt Fitness Calculation

In [8]:
from crypto_analysis.vectorbt_optimizer import VectorbtFitness, calculate_fitness
from crypto_analysis.indicator_optimizer.indicators import INDICATOR_REGISTRY
import pandas as pd

# Get RSI indicator
rsi = INDICATOR_REGISTRY["RSI"]()

# Calculate indicator first, then generate signals
df_with_rsi = rsi.calculate_indicator(df.copy())
entries = rsi.generate_entry_signal(df_with_rsi)
exits = rsi.generate_exit_signal(df_with_rsi)

print(f"Entry signals: {entries.sum()} ({entries.mean():.2%})")
print(f"Exit signals: {exits.sum()} ({exits.mean():.2%})")

# Calculate fitness
fitness = calculate_fitness(df['close'], entries, exits)
print(f"\nFitness: {fitness}")

Entry signals: 415 (4.51%)
Exit signals: 500 (5.43%)

Fitness: FitnessResult(return=0.1154, trades=36, win_rate=58.33%, sharpe=0.48)


## 4. Test Multiple Indicators

In [9]:
# Test a few different indicators
indicators_to_test = ["RSI", "MACD", "BBANDS"]

for ind_name in indicators_to_test:
    try:
        result = optimize_indicator(ind_name, "BTC", n_jobs=2)
        print(f"{ind_name}: score={result.score:.4f}, params={result.best_params}")
    except Exception as e:
        print(f"{ind_name}: ERROR - {e}")

crypto_analysis.vectorbt_optimizer.optimizer - INFO - Optimizing RSI: 12789 combinations
crypto_analysis.vectorbt_optimizer.optimizer - INFO - Using tpe sampler with 1000 trials
crypto_analysis.vectorbt_optimizer - INFO - Optimized RSI for BTC: score=0.6523, params={'timeperiod': 30, 'entry_constant': 30.00946546186002, 'exit_constant': 62.8784064676496}
crypto_analysis.vectorbt_optimizer.optimizer - INFO - Optimizing MACD: 4576 combinations
crypto_analysis.vectorbt_optimizer.optimizer - INFO - Using tpe sampler with 1000 trials


RSI: score=0.6523, params={'timeperiod': 30, 'entry_constant': 30.00946546186002, 'exit_constant': 62.8784064676496}


crypto_analysis.vectorbt_optimizer - INFO - Optimized MACD for BTC: score=0.0000, params={'fastperiod': 18, 'slowperiod': 18, 'signalperiod': 7}
crypto_analysis.vectorbt_optimizer.optimizer - INFO - Optimizing BBANDS: 369 combinations
crypto_analysis.vectorbt_optimizer.optimizer - INFO - Using grid sampler with 369 trials


MACD: score=0.0000, params={'fastperiod': 18, 'slowperiod': 18, 'signalperiod': 7}


crypto_analysis.vectorbt_optimizer - INFO - Optimized BBANDS for BTC: score=0.3448, params={'timeperiod': 29, 'nbdevup': 3.0, 'nbdevdn': 1.0, 'entry_factor': 1.0, 'exit_factor': 0.95}


BBANDS: score=0.3448, params={'timeperiod': 29, 'nbdevup': 3.0, 'nbdevdn': 1.0, 'entry_factor': 1.0, 'exit_factor': 0.95}


## 5. Test Batch Optimization (Small Scale)

In [13]:
from crypto_analysis.vectorbt_optimizer import optimize_all

# Small test: 2 symbols, 2 indicators
results = optimize_all(
    symbols=["BTC", "ETH"],
    indicators=["RSI", "MACD"],
    data_dir="../data/binance",
    output_dir="../output/test_run",
    config_path="../config.json",
    n_processes=2,
    n_jobs_optuna=2,
    export_csv=True,
    export_params_json=True
)

print(f"\nResults for {len(results)} symbols:")
for symbol, df_result in results.items():
    print(f"  {symbol}: {df_result.shape[0]} rows, {df_result.shape[1]} columns")

crypto_analysis.vectorbt_optimizer.parallel_runner - INFO - Running optimization for 2 cryptos, 2 indicators
crypto_analysis.vectorbt_optimizer.output_builder - INFO - Exported ..\output\test_run\BTC_optimized.csv
crypto_analysis.vectorbt_optimizer.output_builder - INFO - Exported ..\output\test_run\BTC_params.json
crypto_analysis.vectorbt_optimizer.parallel_runner - INFO - Completed BTC
crypto_analysis.vectorbt_optimizer.output_builder - INFO - Exported ..\output\test_run\ETH_optimized.csv
crypto_analysis.vectorbt_optimizer.output_builder - INFO - Exported ..\output\test_run\ETH_params.json
crypto_analysis.vectorbt_optimizer.parallel_runner - INFO - Completed ETH
crypto_analysis.vectorbt_optimizer.parallel_runner - INFO - Exported combined params to ..\output\test_run\all_params.json
crypto_analysis.vectorbt_optimizer - INFO - Optimization complete. Processed 2 symbols.



Results for 2 symbols:
  BTC: 9184 rows, 15 columns
  ETH: 9184 rows, 15 columns


In [14]:
# Check output files
from pathlib import Path

output_dir = Path("../output/test_run")
if output_dir.exists():
    files = list(output_dir.glob("*"))
    print(f"Output files ({len(files)}):")
    for f in files:
        print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

Output files (5):
  all_params.json (1.8 KB)
  BTC_optimized.csv (1202.3 KB)
  BTC_params.json (0.8 KB)
  ETH_optimized.csv (1554.1 KB)
  ETH_params.json (0.9 KB)


In [15]:
# View sample output
if "BTC" in results:
    df_btc = results["BTC"]
    print("BTC output columns:")
    print(df_btc.columns.tolist())
    print("\nSample data:")
    display(df_btc.head())

BTC output columns:
['date', 'open', 'high', 'low', 'close', 'volume', 'RSI_entry', 'RSI_exit', 'RSI_rsi', 'MACD_entry', 'MACD_exit', 'MACD_macd', 'MACD_macdsignal', 'MACD_macdhist', 'tradeable']

Sample data:


,date,open,high,low,close,volume,RSI_entry,RSI_exit,RSI_rsi,MACD_entry,MACD_exit,MACD_macd,MACD_macdsignal,MACD_macdhist,tradeable
0,2025-01-01 00:00:00+00:00,93576.00,94509.42,93489.03,94401.14,755.99010,False,False,NaN,False,False,NaN,NaN,NaN,hold
1,2025-01-01 01:00:00+00:00,94401.13,94408.72,93578.77,93607.74,586.53456,False,False,NaN,False,False,NaN,NaN,NaN,hold
2,2025-01-01 02:00:00+00:00,93607.74,94105.12,93594.56,94098.91,276.78045,False,False,NaN,False,False,NaN,NaN,NaN,trade
3,2025-01-01 03:00:00+00:00,94098.90,94098.91,93728.22,93838.04,220.99302,False,False,NaN,False,False,NaN,NaN,NaN,trade
4,2025-01-01 04:00:00+00:00,93838.04,93838.04,93500.00,93553.91,279.46909,False,False,NaN,False,False,NaN,NaN,NaN,trade


## 6. View Params JSON

In [16]:
import json

params_file = Path("../output/test_run/BTC_params.json")
if params_file.exists():
    with open(params_file) as f:
        params = json.load(f)
    print("BTC optimization params:")
    print(json.dumps(params, indent=2))

BTC optimization params:
{
  "RSI": {
    "best_params": {
      "timeperiod": 30,
      "entry_constant": 29.91598170339633,
      "exit_constant": 63.02485602473086
    },
    "score": 0.6741227808892538,
    "type": "tpe",
    "n_trials": 1000,
    "fitness": {
      "total_return": 0.6741227808892538,
      "num_trades": 14,
      "win_rate": 0.7857142857142857,
      "sharpe_ratio": 2.1480014347971084,
      "max_drawdown": -0.17219360468075373
    }
  },
  "MACD": {
    "best_params": {
      "fastperiod": 15,
      "slowperiod": 15,
      "signalperiod": 13
    },
    "score": 0.0,
    "type": "tpe",
    "n_trials": 1000,
    "fitness": {
      "total_return": 0.0,
      "num_trades": 0,
      "win_rate": 0.0,
      "sharpe_ratio": Infinity,
      "max_drawdown": 0.0
    }
  }
}
